# Phase 2: Fused KAN Kernel Testing

Test the fused CUDA kernel (`FusedKanLayer`) that replaces the 7-op KAN graph
with a single kernel launch.

**What to verify:**
1. IR dump shows `FusedKanLayer(grid=5, order=3)` instead of separate ops
2. Training loss matches Phase 1 unfused results (numerical equivalence)
3. Training is faster (fewer kernel launches per step)

**Runtime**: GPU (T4 or better). Go to Runtime > Change runtime type > GPU.

## 1. Setup

In [ ]:
%%bash
# Install Rust
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
fi
source $HOME/.cargo/env
rustc --version && cargo --version

: 

In [ ]:
%%bash
source $HOME/.cargo/env

if [ ! -d /content/bullet ]; then
    git clone https://github.com/y0sif/bullet.git /content/bullet
fi
cd /content/bullet
git fetch origin main
git reset --hard origin/main
git log --oneline -5

## 2. Download Training Data

In [ ]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test77.binpack ]; then
    echo "Downloading test77 binpack (~1.3 GB compressed)..."
    wget -q -O test77.binpack.zst \
        "https://huggingface.co/datasets/linrock/test77/resolve/main/test77-2022-01-jan-2tb7p.binpack.zst"
    echo "Decompressing..."
    zstd -d test77.binpack.zst -o test77.binpack --rm
    echo "Done!"
fi

ls -lh test77.binpack

## 3. Build

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet

nvidia-smi | head -4
echo "---"

echo "Building kan_simple (fused) and kan_baseline..."
cargo build --release --example kan_simple --example kan_baseline 2>&1
echo "Build complete!"

## 4. Verify Fusion Pass (IR Dump)

Run `kan_simple` briefly — it prints the IR graph before training starts.
Look for `FusedKanLayer(grid=5, order=3)` in the output.

If you see separate `BSplineBasis`, `Matmul`, `Sigmoid`, `Concat`, `PairwiseMul` nodes
instead, the fusion pass didn't fire.

In [ ]:
import subprocess, os, re
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

# Run kan_simple, capture first ~50 lines (IR dump happens before training)
proc = subprocess.Popen(
    ["cargo", "run", "--release", "--example", "kan_simple"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd="/content/bullet", text=True, bufsize=1
)

ir_lines = []
training_started = False
for line in proc.stdout:
    clean = strip_ansi(line).rstrip()
    ir_lines.append(clean)
    # Stop after first superbatch line (IR dump is before training)
    if 'superbatch' in clean.lower() and 'running loss' in clean.lower():
        training_started = True
        break

proc.terminate()
proc.wait()

# Print IR
print("=== IR Graph Dump ===")
for line in ir_lines:
    print(line)

# Check for fusion
ir_text = '\n'.join(ir_lines)
if 'FusedKanLayer' in ir_text:
    print("\n>>> FUSION PASS FIRED: FusedKanLayer detected in IR")
elif 'BSplineBasis' in ir_text:
    print("\n>>> WARNING: Fusion pass did NOT fire — still using unfused BSplineBasis ops")
else:
    print("\n>>> Could not determine fusion status from IR output")

## 5. Train with Fused Kernel

Full training run (40 superbatches). Compare loss and timing to Phase 1 unfused results.

In [ ]:
import subprocess, sys, os, time
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

cmd = ["cargo", "run", "--release", "--example", "kan_simple"]
start = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        cwd="/content/bullet", text=True, bufsize=1)
log = open("/content/fused_kan_log.txt", "w")
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
    log.write(line)
log.close()
proc.wait()
elapsed = time.time() - start
print(f"\nTotal time: {elapsed:.1f}s")
print(f"Exited with code {proc.returncode}")

## 6. Compare: Fused vs Unfused

Load the Phase 1 (unfused) KAN log from Drive and compare loss curves.
If no Drive log available, run the baseline example for comparison.

In [ ]:
import re
import matplotlib.pyplot as plt

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

def parse_bullet_log(path):
    """Parse Bullet training log: returns list of (superbatch, loss, time_secs)."""
    results = []
    with open(path) as f:
        for line in f:
            clean = strip_ansi(line)
            m = re.search(r'superbatch\s+(\d+)\s+\|\s+time\s+([\d.]+)s\s+\|.*?running loss\s+([\d.]+)', clean)
            if m:
                results.append((int(m.group(1)), float(m.group(3)), float(m.group(2))))
    return results

# Parse fused results
fused = parse_bullet_log('/content/fused_kan_log.txt')

# Try loading Phase 1 unfused results from Drive
unfused = []
unfused_path = '/content/drive/MyDrive/kanue-bullet/kan_log.txt'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    unfused = parse_bullet_log(unfused_path)
    print(f'Loaded Phase 1 unfused log: {len(unfused)} superbatches')
except Exception as e:
    print(f'Could not load Phase 1 log from Drive: {e}')
    print('Will show fused results only.')

if fused:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Loss comparison ---
    ax = axes[0]
    ax.plot([x[0] for x in fused], [x[1] for x in fused],
            label='KAN (Fused)', linewidth=2, color='#2196F3')
    if unfused:
        ax.plot([x[0] for x in unfused], [x[1] for x in unfused],
                label='KAN (Unfused, Phase 1)', linewidth=2, color='#FF9800', linestyle='--')
    ax.set_xlabel('Superbatch')
    ax.set_ylabel('Loss')
    ax.set_title('Loss: Fused vs Unfused KAN')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Timing comparison ---
    ax = axes[1]
    ax.plot([x[0] for x in fused], [x[2] for x in fused],
            label='Fused', linewidth=2, color='#2196F3')
    if unfused:
        ax.plot([x[0] for x in unfused], [x[2] for x in unfused],
                label='Unfused (Phase 1)', linewidth=2, color='#FF9800', linestyle='--')
    ax.set_xlabel('Superbatch')
    ax.set_ylabel('Time (seconds)')
    ax.set_title('Time per Superbatch')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/fused_vs_unfused.png', dpi=150)
    plt.show()

    # Print summary
    fused_final = fused[-1][1]
    fused_avg_time = sum(x[2] for x in fused) / len(fused)
    print(f'\nFused KAN:')
    print(f'  Final loss:           {fused_final:.6f}')
    print(f'  Avg time/superbatch:  {fused_avg_time:.1f}s')

    if unfused:
        unfused_final = unfused[-1][1]
        unfused_avg_time = sum(x[2] for x in unfused) / len(unfused)
        loss_diff = abs(fused_final - unfused_final) / unfused_final * 100
        speedup = unfused_avg_time / fused_avg_time if fused_avg_time > 0 else 0
        print(f'\nUnfused KAN (Phase 1):')
        print(f'  Final loss:           {unfused_final:.6f}')
        print(f'  Avg time/superbatch:  {unfused_avg_time:.1f}s')
        print(f'\nDifference:')
        print(f'  Loss delta:  {loss_diff:.2f}% {"(GOOD: ~same)" if loss_diff < 1 else "(WARNING: diverged)"}')
        print(f'  Speedup:     {speedup:.2f}x')
else:
    print('No fused training results found.')

## 7. Save Results

In [ ]:
import shutil, os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = Path('/content/drive/MyDrive/kanue-bullet')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

for src in ['/content/fused_kan_log.txt', '/content/fused_vs_unfused.png']:
    if os.path.exists(src):
        shutil.copy2(src, DRIVE_BASE / os.path.basename(src))
        print(f'Copied {src}')

# Copy checkpoints
ckpt_dir = Path('/content/bullet/checkpoints')
if ckpt_dir.exists():
    dest = DRIVE_BASE / 'checkpoints-fused'
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(ckpt_dir, dest)
    print(f'Copied checkpoints -> {dest}')